# 09 — Training source-quality sensitivity (optional research experiment)

**Question:** how much do the frozen model's validation results change when additional high-priority suspected training prices are excluded?

Run the final review cell in 03 first, then **Run All** here. No path or mode edits are needed. Your current shortlist has 58 additional priority-1 cases; the actual count is checked and reported from this run's review table, not hard-coded.

This is a **retrospective, exploratory holdout-training analysis**, not real-time cleaning or fresh independent validation. The shortlist was reviewed after test exposure and uses neighbouring training observations through December 2025. Do not apply it to earlier expanding folds. Suspected values are excluded, never replaced or declared confirmed errors.

**Fixed:** the selected model family/settings, preprocessing strategy, original score cutoff, and all original validation features, labels and rows. **Rebuilt:** training weekly aggregates, exact-week features, eligibility and labels. Thus this measures the combined effect of changed training measurements and support, not simply deleting modelling rows. Validation history remains as originally prepared, deliberately isolating a training-only intervention.

The notebook never loads or hashes test data, runs a candidate search, selects a new cutoff, or changes 01–08. A matching supplementary cache is reused. This does not automatically promote a model to the dashboard.

In [1]:
# Open this notebook from the FYP folder or its research subfolder.
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open the FYP project folder before running this research notebook.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import hashlib
import json
from pathlib import Path
from importlib import reload, metadata
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
import pricecatcher_config as cfg
cfg = reload(cfg)
if cfg.EXPERIMENT != "holdout":
    raise ValueError("This retrospective review is for holdout only, not early folds.")

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

model_dir = next((p for p in cfg.MODEL_DIRS if (p / "best_model.joblib").is_file()), None)
if model_dir is None:
    raise FileNotFoundError("Complete Notebook 06 first; this notebook does not train a replacement main experiment.")
manifest = json.loads((model_dir / "model_manifest.json").read_text())
selection = json.loads((model_dir / "selection.json").read_text())
if manifest.get("run_status") != "completed" or selection != manifest["selection"]:
    raise ValueError("A matching completed main experiment is required.")
if file_hash(model_dir / "best_model.joblib") != manifest["model_sha256"]:
    raise ValueError("Frozen model changed. Restore its original file.")
required_packages = ["scikit-learn", "joblib"]
if selection["family"] == "LightGBM":
    required_packages.append("lightgbm")
if selection["family"] == "XGBoost":
    required_packages += [p for p in ["xgboost", "xgboost-cpu"] if p in manifest["package_versions"]]
for package in required_packages:
    if metadata.version(package) != manifest["package_versions"][package]:
        raise ValueError(f"Restore the saved model's {package} version before loading it.")
folder = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling"
# Verify ONLY development inputs: no test file is opened, including for hashing.
for name in ["experiment_settings.json", "train.parquet", "validation.parquet"]:
    if file_hash(folder / name) != manifest["input_sha256"][f"holdout/{name}"]:
        raise ValueError(f"Frozen {name} changed. Restore the original input.")
settings = json.loads((folder / "experiment_settings.json").read_text())
training_end = pd.Timestamp(settings["train_end"])
validation_end = pd.Timestamp(settings["validation_end"])
features = settings["feature_columns"]
keys = cfg.GROUP_KEYS + ["week_start"]
source_manifest = cfg.verify_source_snapshot()
cleaning_path = cfg.STAGE3_DIR / "cleaning_summary.json"
cleaning = json.loads(cleaning_path.read_text())
if cleaning["preparation"] != settings["preparation"] or settings["preparation"] != {
        **cfg.PREPARATION_SETTINGS, "source_manifest_sha256": file_hash(cfg.STAGE1_DIR / "source_manifest.json"),
        "source_snapshot_sha256": source_manifest["source_snapshot_sha256"]}:
    raise ValueError("This sensitivity needs the matching current preparation and source, without metadata repairs.")
review_path = cfg.STAGE3_DIR / "training_source_context.csv"
if not review_path.exists():
    raise FileNotFoundError("Run only the final source-review cell in Notebook 03 first.")
review = pd.read_csv(review_path, parse_dates=["date", "previous_date", "following_date"])
if (review.empty or review.duplicated(["date", "premise_code", "item_code"]).any()
    or not review["source_snapshot_sha256"].eq(source_manifest["source_snapshot_sha256"]).all()
    or not review["training_end"].eq(settings["train_end"]).all()
    or not review["date"].le(training_end).all()):
    raise ValueError("Review must contain unique training cases from this exact source/cutoff.")
# Recompute priority 1 rather than trusting an edited priority column.
ratios = review[["previous_price", "following_price"]].rdiv(review["price"], axis=0)
large = ((ratios >= 10 - 1e-8) | (ratios <= 0.1 + 1e-8)).all(axis=1)
nearby = ((review.date - review.previous_date).dt.days.between(1, 7)
          & (review.following_date - review.date).dt.days.between(1, 7))
stable = (review.previous_price.gt(0) & review.following_price.gt(0)
          & (review.previous_price - review.following_price).abs().le(0.10 * review.previous_price))
priority_one = large & nearby & stable
if not priority_one.equals(review["review_priority"].eq(1)):
    raise ValueError("Review priorities disagree with the declared rule. Regenerate the final cell in 03.")
shortlist = review.loc[priority_one & review.existing_policy.ne("documented conservative exclusion")].copy()
if shortlist.empty:
    raise ValueError("No additional priority-1 training cases; there is no exclusion experiment to fit.")
for date_column in ["previous_date", "following_date"]:
    if shortlist[date_column].gt(training_end).any():
        raise ValueError("Review neighbours extend beyond training.")

tracked_paths = [model_dir / name for name in ["best_model.joblib", "model_manifest.json", "selection.json"]]
tracked_paths += [folder / name for name in ["experiment_settings.json", "train.parquet", "validation.parquet"]]
tracked_paths += [review_path, cleaning_path, cfg.STAGE3_MODEL_READY_PARQUET, cfg.STAGE1_DIR / "source_manifest.json"]
input_hashes = {str(p.relative_to(cfg.PROJECT_ROOT)): file_hash(p) for p in tracked_paths}
# Hash executable cells only; saving notebook outputs does not create another experiment.
notebook_path = cfg.notebook_path("09_source_quality_sensitivity.ipynb")
notebook_code = "\n".join("".join(c["source"]) for c in json.loads(notebook_path.read_text())["cells"] if c["cell_type"] == "code")
protocol = {"schema_version": 1, "status": "post-hoc exploratory validation sensitivity",
    "candidate": selection["candidate"], "cutoff": selection["score_cutoff"],
    "intervention": "exclude additional priority-1 training records; rebuild training only",
    "code_sha256": hashlib.sha256(notebook_code.encode()).hexdigest(),
    "input_sha256": input_hashes,
    "packages": {p: metadata.version(p) for p in list(dict.fromkeys(["pandas", "numpy"] + required_packages))}}
experiment_id = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode()).hexdigest()[:16]
output_dir = cfg.PROJECT_ROOT / "09_source_quality_sensitivity" / experiment_id
output_dir.mkdir(parents=True, exist_ok=True)
print("Source run:", cfg.PROJECT_ROOT)
print("Additional suspected records:", len(shortlist), "| Output:", output_dir)
print("Validation comparison only; no test access, cutoff search or dashboard replacement.")

Source run: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240
Additional suspected records: 58 | Output: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/09_source_quality_sensitivity/4dc2973d322e3db8
Validation comparison only; no test access, cutoff search or dashboard replacement.


## Rebuild training measurements with the same rules as 04–05

Before applying exclusions, the reconstruction must reproduce the frozen training features, labels and row keys. This is a check against unintended changes in the feature builder. Following exclusions, report gained/lost rows, changed labels and changed features. The 10% target and numerical tolerance remain fixed.

In [2]:
def training_table(daily):
    """Same weekly median/coverage and exact-calendar history as Notebooks 04–05."""
    daily = daily.copy()
    daily["week_start"] = daily.date - pd.to_timedelta(daily.date.dt.weekday, unit="D")
    weekly = daily.groupby(keys).agg(weekly_median_price=("price", "median"),
        weekly_std_price=("price", "std"), record_count=("price", "size"),
        premise_count=("premise_code", "nunique"), observed_days=("date", "nunique")).reset_index()
    weekly = weekly.loc[(weekly.week_start >= pd.Timestamp(cfg.START_MONTH + "-01"))
        & ((weekly.week_start + pd.Timedelta(days=6)) <= training_end)]
    parts = []
    for _, series in weekly.groupby(cfg.GROUP_KEYS, sort=False):
        grid = series.set_index("week_start").sort_index()
        grid = grid.reindex(pd.date_range(grid.index.min(), grid.index.max(), freq="W-MON"))
        price = grid.weekly_median_price
        for lag in [1, 2, 4]:
            grid[f"lag_{lag}w_price"] = price.shift(lag)
        for window in [4, 8]:
            grid[f"rolling_mean_{window}w"] = price.shift(1).rolling(window, min_periods=2).mean()
            grid[f"rolling_std_{window}w"] = price.shift(1).rolling(window, min_periods=2).std()
        grid["history_count_4w"] = price.shift(1).rolling(4, min_periods=1).count()
        grid["weekly_price_change_pct"] = (price / price.shift(1) - 1) * 100
        grid["change"] = (price.shift(-1) / price - 1) * 100
        current_ok = grid.record_count.ge(cfg.MIN_RECORDS) & grid.premise_count.ge(cfg.MIN_PREMISES)
        next_ok = grid.record_count.shift(-1).ge(cfg.MIN_RECORDS) & grid.premise_count.shift(-1).ge(cfg.MIN_PREMISES)
        eligible = current_ok & next_ok & grid.lag_1w_price.notna() & grid.history_count_4w.ge(cfg.MIN_HISTORY_4W)
        grid["surge_label"] = cfg.surge_mask(grid.change, settings["surge_threshold_pct"],
            settings["preparation"]["label_tolerance_pct"]).astype(int)
        part = grid.loc[price.notna() & eligible].copy()
        part.index.name = "week_start"
        parts.append(part.reset_index())
    result = pd.concat(parts, ignore_index=True)
    result["next_week_end"] = result.week_start + pd.Timedelta(days=13)
    result = result.loc[result.next_week_end.le(training_end)].copy()
    result["month"] = result.week_start.dt.month
    week_number = result.week_start.dt.isocalendar().week.astype(float)
    result["week_sin"] = np.sin(2 * np.pi * week_number / 52.1775)
    result["week_cos"] = np.cos(2 * np.pi * week_number / 52.1775)
    return result.sort_values(keys).reset_index(drop=True)

def model_inputs(frame):
    X = frame[features].copy()
    for column in settings["categorical_features"]:
        X[column] = X[column].astype("string").fillna("__missing__").astype(str)
    return X

def metrics(frame, scores):
    y = frame.surge_label.astype(int)
    predicted = scores >= selection["score_cutoff"]
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    return {"rows": len(y), "positives": int(y.sum()),
        "average_precision": average_precision_score(y, scores) if y.nunique() == 2 else np.nan,
        "precision": precision_score(y, predicted, zero_division=0),
        "recall": recall_score(y, predicted, zero_division=0) if y.sum() else np.nan,
        "f1": f1_score(y, predicted, zero_division=0) if y.sum() else np.nan,
        "true_alerts": int(tp), "false_alerts": int(fp), "missed_surges": int(fn), "true_negatives": int(tn)}

## Fit once and compare on the unchanged validation population

The original pipeline is cloned: preprocessing is refitted on the rebuilt training data, and estimator parameters stay fixed. Built-in balanced class weights follow their original training-only policy; explicit numeric weights remain fixed. Both versions use the original validation-selected cutoff. No alternative cutoff is searched.

A complete matching cache reopens without fitting. If an interrupted run has no completed manifest, rerunning rebuilds only this supplementary experiment. A changed code/input fingerprint creates another isolated folder, never overwriting earlier experiments.

In [3]:
cache_path = output_dir / "sensitivity_manifest.json"
artifact_names = ["sensitivity_model.joblib", "excluded_training_records.csv", "training_impact.csv",
    "validation_comparison.csv", "validation_group_comparison.csv", "validation_predictions.parquet"]
cache = json.loads(cache_path.read_text()) if cache_path.exists() else None
if cache is not None:
    if cache["protocol"] != protocol or set(cache["output_sha256"]) != set(artifact_names):
        raise ValueError("Supplementary manifest does not match this experiment.")
    if any(not (output_dir / name).is_file() or file_hash(output_dir / name) != cache["output_sha256"][name]
           for name in artifact_names):
        raise ValueError("Supplementary cache changed. Restore this experiment's original artifacts.")
    print("Reusing completed sensitivity results; no fitting.")
else:
    original_train = pd.read_parquet(folder / "train.parquet").sort_values(keys).reset_index(drop=True)
    validation = pd.read_parquet(folder / "validation.parquet").sort_values(keys).reset_index(drop=True)
    if (validation.week_start + pd.Timedelta(days=6)).le(training_end).any() or validation.next_week_end.gt(validation_end).any():
        raise ValueError("Validation dates do not respect the frozen chronological boundary.")
    print("Rebuilding original training measurements for the consistency check...")
    daily = pd.read_parquet(cfg.STAGE3_MODEL_READY_PARQUET, columns=cfg.GROUP_KEYS + ["date", "premise_code", "price"],
        filters=[("date", "<=", training_end)])
    daily["date"] = pd.to_datetime(daily.date)
    rebuilt_original = training_table(daily)
    comparison_columns = list(dict.fromkeys(keys + features + ["surge_label", "next_week_end"]))
    pd.testing.assert_frame_equal(original_train[comparison_columns], rebuilt_original[comparison_columns],
        check_dtype=False, check_exact=False, rtol=1e-9, atol=1e-9)
    print("Original reconstruction matches. Rebuilding after the shortlisted exclusions...")
    daily_keys = ["date", "premise_code", "item_code"]
    matched = shortlist.merge(daily[daily_keys + ["price"]], on=daily_keys, how="left",
        validate="one_to_one", suffixes=("", "_daily"))
    if matched.price_daily.isna().any() or not np.isclose(matched.price, matched.price_daily, rtol=0, atol=1e-12).all():
        raise ValueError("Shortlisted prices do not match the retained training daily data.")
    daily = daily.merge(shortlist[daily_keys].assign(exclude=True), on=daily_keys, how="left", validate="one_to_one")
    cleaned_train = training_table(daily.loc[daily.exclude.ne(True)].drop(columns="exclude"))
    if cleaned_train.empty or cleaned_train.surge_label.nunique() != 2:
        raise ValueError("Rebuilt training needs supported observations from both classes.")
    common = original_train.merge(cleaned_train, on=keys, suffixes=("_original", "_sensitivity"), validate="one_to_one")
    original_keys = pd.MultiIndex.from_frame(original_train[keys])
    cleaned_keys = pd.MultiIndex.from_frame(cleaned_train[keys])
    changed_features = np.zeros(len(common), dtype=bool)
    for column in settings["numeric_features"]:
        changed_features |= ~np.isclose(common[column + "_original"].to_numpy(dtype=float),
            common[column + "_sensitivity"].to_numpy(dtype=float), rtol=1e-9, atol=1e-9, equal_nan=True)
    impact = pd.DataFrame([{"excluded_daily_records": len(shortlist), "original_training_rows": len(original_train),
        "sensitivity_training_rows": len(cleaned_train), "common_training_rows": len(common),
        "lost_training_rows": int((~original_keys.isin(cleaned_keys)).sum()),
        "gained_training_rows": int((~cleaned_keys.isin(original_keys)).sum()),
        "changed_labels_on_common_rows": int(common.surge_label_original.ne(common.surge_label_sensitivity).sum()),
        "common_rows_with_changed_features": int(changed_features.sum()),
        "original_training_surges": int(original_train.surge_label.sum()),
        "sensitivity_training_surges": int(cleaned_train.surge_label.sum())}])
    original_model = joblib.load(model_dir / "best_model.joblib")
    sensitivity_model = clone(original_model)
    print("Fitting the supplementary model with fixed parameters and cutoff...")
    sensitivity_model.fit(model_inputs(cleaned_train), cleaned_train.surge_label.astype(int))
    scores = {"frozen_original": original_model.predict_proba(model_inputs(validation))[:, 1],
              "training_exclusion_sensitivity": sensitivity_model.predict_proba(model_inputs(validation))[:, 1]}
    comparison = pd.DataFrame([{"version": name, "cutoff": selection["score_cutoff"], **metrics(validation, values)}
        for name, values in scores.items()])
    group_rows = []
    for group, part in validation.groupby("project_group"):
        for name, values in scores.items():
            group_rows.append({"project_group": group, "version": name, **metrics(part, values[part.index])})
    predictions = validation[keys + ["next_week_end", "surge_label"]].copy()
    for name, values in scores.items():
        predictions[name + "_score"] = values
    joblib.dump(sensitivity_model, output_dir / "sensitivity_model.joblib")
    shortlist.to_csv(output_dir / "excluded_training_records.csv", index=False)
    impact.to_csv(output_dir / "training_impact.csv", index=False)
    comparison.to_csv(output_dir / "validation_comparison.csv", index=False)
    pd.DataFrame(group_rows).to_csv(output_dir / "validation_group_comparison.csv", index=False)
    predictions.to_parquet(output_dir / "validation_predictions.parquet", index=False)
    # Check original development inputs/model/source evidence are still byte-identical.
    if any(file_hash(cfg.PROJECT_ROOT / name) != value for name, value in input_hashes.items()):
        raise ValueError("An upstream input changed during execution; results are not complete.")
    cache = {"protocol": protocol, "test_accessed": False, "cutoff_retuned": False,
        "dashboard_model_replaced": False, "output_sha256": {name: file_hash(output_dir / name) for name in artifact_names}}
    temporary = cache_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(cache, indent=2))
    temporary.replace(cache_path)

print("Training measurement/support changes:")
display(pd.read_csv(output_dir / "training_impact.csv"))
print("Validation metrics at the same frozen cutoff:")
comparison = pd.read_csv(output_dir / "validation_comparison.csv")
display(comparison)
display(pd.read_csv(output_dir / "validation_group_comparison.csv"))
delta = comparison.set_index("version").loc["training_exclusion_sensitivity", ["average_precision", "precision", "recall", "f1"]] - comparison.set_index("version").loc["frozen_original", ["average_precision", "precision", "recall", "f1"]]
display(delta.rename("sensitivity_minus_original").to_frame())
print("Saved:", output_dir)
print("Exploratory sensitivity, not a confirmed correction or independent performance claim.")

Rebuilding original training measurements for the consistency check...
Original reconstruction matches. Rebuilding after the shortlisted exclusions...
Fitting the supplementary model with fixed parameters and cutoff...
Training measurement/support changes:


,excluded_daily_records,original_training_rows,sensitivity_training_rows,common_training_rows,lost_training_rows,gained_training_rows,changed_labels_on_common_rows,common_rows_with_changed_features,original_training_surges,sensitivity_training_surges
0,58,195664,195664,195664,0,0,2,144,20048,20048


Validation metrics at the same frozen cutoff:


,version,cutoff,rows,positives,average_precision,precision,recall,f1,true_alerts,false_alerts,missed_surges,true_negatives
0,frozen_original,0.60065,43594,3912,0.223372,0.221820,0.657464,0.331721,2572,9023,1340,30659
1,training_exclusion_sensitivity,0.60065,43594,3912,0.226298,0.223737,0.658742,0.334025,2577,8941,1335,30741


,project_group,version,rows,positives,average_precision,precision,recall,f1,true_alerts,false_alerts,missed_surges,true_negatives
0,Chicken,frozen_original,4884,25,0.132102,0.145833,0.280000,0.191781,7,41,18,4818
1,Chicken,training_exclusion_sensitivity,4884,25,0.105835,0.157895,0.240000,0.190476,6,32,19,4827
2,Cooking Oil,frozen_original,2178,1,0.000548,0.000000,0.000000,0.000000,0,0,1,2177
3,Cooking Oil,training_exclusion_sensitivity,2178,1,0.000482,0.000000,0.000000,0.000000,0,0,1,2177
4,Eggs,frozen_original,8892,53,0.076957,0.138462,0.169811,0.152542,9,56,44,8783
5,Eggs,training_exclusion_sensitivity,8892,53,0.077318,0.117647,0.150943,0.132231,8,60,45,8779
6,Flour,frozen_original,1110,15,0.238534,0.250000,0.533333,0.340426,8,24,7,1071
7,Flour,training_exclusion_sensitivity,1110,15,0.228731,0.240000,0.400000,0.300000,6,19,9,1076
8,Rice,frozen_original,384,2,0.514706,1.000000,0.500000,0.666667,1,0,1,382
9,Rice,training_exclusion_sensitivity,384,2,0.524390,1.000000,0.500000,0.666667,1,0,1,382


,sensitivity_minus_original
average_precision,0.002926
precision,0.001917
recall,0.001278
f1,0.002303


Saved: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/09_source_quality_sensitivity/4dc2973d322e3db8
Exploratory sensitivity, not a confirmed correction or independent performance claim.


## Interpret before making another change

- AP measures ranking independently of the fixed score cutoff. Precision/recall/F1 also reflect changes in score distribution; this is not a calibration study.
- An improvement suggests sensitivity to the reviewed training observations. It does not confirm wrong prices, causation or dependable warnings. A deterioration is also useful evidence.
- Report changed training labels and eligibility as well as model metrics. Validation coverage is identical by design; training geography can still change.
- This comparison combines the selected exclusion policy with the existing frozen model choice; it is not a new expanding-window selection study. Do not repeatedly tune review thresholds or exclusions against these validation results.
- Keep the existing main experiment and dashboard model. A future independently evaluated version needs an untouched later period and a documented cleaning protocol available at prediction time.

Share `training_impact.csv`, `validation_comparison.csv` and `validation_group_comparison.csv` for interpretation. No full source upload or another 01–08 run is needed.